# Machine Learning #02 — Classification & Regression with PyTorch 🔥

Last time we trained a **binary classifier**. Today we take two steps further and show that
**classification and regression share a training workflow**, while their targets, losses,
and evaluation metrics differ.

We will build **two networks** with PyTorch:

1. 🏷️ **Classification** — recognise hand-written digits (**MNIST**, 10 classes).
2. 📈 **Regression** — predict a car's fuel economy (**Auto MPG**, a continuous number).

## 🎯 The big idea: what changes between the two tasks?

We reuse the training workflow from lesson #1. The output head, targets, loss,
metrics, and data preparation must match the task. Keep this table in mind for the whole lesson — we will refer back to it:

| | 🏷️ **Classification** (MNIST) | 📈 **Regression** (Auto MPG) |
|---|---|---|
| **Question** | *Which class?* (a category) | *How much?* (a number) |
| **Output neurons** | one per class (**10**) | **1** |
| **Output activation** | none in model → `softmax` at eval | **none / linear** |
| **Loss function** | `nn.CrossEntropyLoss` | `nn.MSELoss` |
| **Metric** | accuracy | MAE, RMSE, $R^2$ |
| **Target dtype** | `int64` (class index) | `float32` (real value) |

Both examples reuse Linear layers, activations, Adam, and the same mini-batch training loop.
Layer sizes, regularization, and learning rate can differ. The target shapes here are
`(batch_size,)` for class indices and `(batch_size, 1)` for regression.

## Imports & setup ⚙️

CPU is sufficient for this exercise; a Colab GPU is optional. Select a notebook kernel with
`torch`, a compatible `torchvision`, `numpy`, `pandas`, `matplotlib`, `seaborn`, and `scikit-learn`.
From the repository root, `python -m pip install .` installs the project dependencies.
The first data-loading run requires internet; later runs reuse the local cache.
`torchinfo` is optional and has a fallback below.

In [ ]:
# Optional model summaries: install torchinfo in your kernel environment if desired.
# The notebook also runs without it; no installation cell is required.

In [ ]:
from copy import deepcopy
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

try:
    import torchinfo
except ImportError:
    torchinfo = None
from torchvision import datasets   # MNIST download and local cache

import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             mean_absolute_error, mean_squared_error, r2_score)

sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(1)  # Avoid threading overhead for these small models.

def summarize_model(model, input_size):
    if torchinfo is None:
        print(model)
        print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
    else:
        torchinfo.summary(model, input_size=input_size, device=device, mode="eval")

print("torch version:", torch.__version__)
print("device:", device)

### 🟰 One training loop for both tasks

In lesson #1 we used full-batch updates. Here we use a `DataLoader` to take mini-batch
updates, reducing memory use and giving several parameter updates per epoch. The whole
MNIST subset could fit in memory, but mini-batches are a useful general training pattern.

The same loop handles classification and regression with the corresponding loss function.
Both losses below use `reduction="mean"`. After each epoch, we measure training and validation
loss at the **same parameter state**, with dropout disabled. These evaluation curves make
comparison easier; the stochastic losses during training can differ.

We restore the checkpoint with the lowest validation loss among the completed epochs.
Use validation data for every comparison and activity; reserve test data for the final section.

In [ ]:
def mean_loss(model, loader, criterion):
    model.eval()
    model_device = next(model.parameters()).device
    total, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(model_device), yb.to(model_device)
            outputs = model(xb)
            if isinstance(criterion, nn.MSELoss) and outputs.shape != yb.shape:
                raise ValueError("MSE predictions and targets must have the same shape.")
            total += criterion(outputs, yb).item() * len(xb)
            count += len(xb)
    return total / count


def train_model(model, train_loader, val_loader, criterion, optimizer, n_epochs=15):
    if n_epochs < 1:
        raise ValueError("n_epochs must be positive.")
    if criterion.reduction != "mean":
        raise ValueError("This helper expects a mean-reduced loss.")
    if len(train_loader.dataset) == 0 or len(val_loader.dataset) == 0:
        raise ValueError("Training and validation datasets must be nonempty.")
    model_device = next(model.parameters()).device
    # A separate loader measures all training examples without shuffling or dropout.
    train_eval_loader = DataLoader(train_loader.dataset, batch_size=256, shuffle=False)
    history = {"train_loss": [], "val_loss": [], "best_epoch": None, "best_val_loss": float("inf")}
    best_state = None

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(model_device), yb.to(model_device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(xb)
            if isinstance(criterion, nn.MSELoss) and outputs.shape != yb.shape:
                raise ValueError("MSE predictions and targets must have the same shape.")
            loss = criterion(outputs, yb)
            if not torch.isfinite(loss):
                raise ValueError("Non-finite training loss; check data and learning rate.")
            loss.backward()
            optimizer.step()

        train_loss = mean_loss(model, train_eval_loader, criterion)
        val_loss = mean_loss(model, val_loader, criterion)
        if not np.isfinite([train_loss, val_loss]).all():
            raise ValueError("Non-finite evaluation loss.")
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if val_loss < history["best_val_loss"]:
            history["best_val_loss"] = val_loss
            history["best_epoch"] = epoch + 1
            best_state = deepcopy(model.state_dict())
        if (epoch + 1) % max(1, n_epochs // 10) == 0:
            print(f"epoch {epoch+1:>3}/{n_epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    print(f"Restored epoch {history['best_epoch']} (validation loss {history['best_val_loss']:.4f})")
    return history


def predict_batches(model, features, batch_size=256):
    model.eval()
    model_device = next(model.parameters()).device
    loader = DataLoader(TensorDataset(features), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        return torch.cat([model(xb.to(model_device)).cpu() for (xb,) in loader])

# 🏷️ Part 1 — Classification: MNIST digits

**MNIST** is the classic "hello world" of deep learning: 70,000 grayscale images of hand-written
digits (28×28 pixels), each labelled **0–9**.

### Why a plain (fully-connected) network and not a CNN? 🤔
Convolutional networks (CNNs) exploit the spatial structure of images and we cover them **next lesson**.
Today we deliberately use a simple **MLP** (Multi-Layer Perceptron): we **flatten** each 28×28
image into a vector of **784 numbers** and treat those as ordinary features — exactly like the
sloth measurements last week, just more of them.

In [ ]:
# The cache is relative to the directory from which the notebook is run.
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_full = datasets.MNIST(root=str(DATA_DIR), train=True, download=True)
test_ds = datasets.MNIST(root=str(DATA_DIR), train=False, download=True)

In [ ]:
train_full.data[0]

In [ ]:
# .data -> uint8 pixels in [0, 255], shape [N, 28, 28];  .targets -> class index 0..9
X_all = train_full.data.float() / 255.0     # normalise pixels to [0, 1]  (same idea as scaling features)
y_all = train_full.targets                  # already int64 -> perfect for CrossEntropyLoss

X_test_full = test_ds.data.float() / 255.0
y_test_full = test_ds.targets

print("images:", X_all.shape, "| labels:", y_all.shape, "| pixel range:", X_all.min().item(), "-", X_all.max().item())

In [ ]:
X_all.shape, y_all.shape

In [ ]:
X_test_full.shape, y_test_full.shape

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, X_all[:8], y_all[:8]):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"label: {label.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Train / validation / test split

Choose a reproducible, stratified subset from the official 60,000 training images:
10,000 for training and 2,000 for validation. Keep the official 10,000 test images separate.
The remaining training images are unused for this classroom run.

Dividing pixels by the known constant 255 does not fit any statistics from held-out data.
The `_c` suffix identifies classification data so regression cells cannot overwrite it.

In [ ]:
subset_idx, unused_idx = train_test_split(
    np.arange(len(X_all)), train_size=12000, stratify=y_all.numpy(), random_state=SEED
)
train_idx_c, val_idx_c = train_test_split(
    subset_idx, test_size=2000, stratify=y_all[subset_idx].numpy(), random_state=SEED
)
X_train_c, y_train_c = X_all[train_idx_c], y_all[train_idx_c]
X_val_c, y_val_c = X_all[val_idx_c], y_all[val_idx_c]
X_test_c, y_test_c = X_test_full, y_test_full

train_loader_c = DataLoader(
    TensorDataset(X_train_c, y_train_c), batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader_c = DataLoader(TensorDataset(X_val_c, y_val_c), batch_size=256)
print("train:", len(X_train_c), "| val:", len(X_val_c), "| test:", len(X_test_c))

### 🌳 Baseline first: is a neural network even worth it?

Before deep learning, let's see how a **plain Decision Tree** does. A baseline tells us whether
the extra complexity of a neural network actually pays off.

In [ ]:
tree = DecisionTreeClassifier(random_state=SEED)
tree.fit(X_train_c.reshape(-1, 28*28).numpy(), y_train_c.numpy())
tree_acc = accuracy_score(y_val_c.numpy(), tree.predict(X_val_c.reshape(-1, 28*28).numpy()))
print(f"Decision Tree validation accuracy: {tree_acc:.3f}")

### 🧠 Build the first architecture (`nn.Module`)

We define the forward computation explicitly in an **`nn.Module` subclass**.
Its ingredients are:

- **multiple hidden layers** (784 → 256 → 128 → 10),
- **`ReLU`** activations,
- **`Dropout`** for regularisation (randomly zeros activation elements during training and rescales the survivors).

> 💡 The output layer has **10 neurons** (one per digit) and **no softmax inside the model**.
> `nn.CrossEntropyLoss` expects raw **logits** and combines log-softmax and negative log-likelihood internally — cleaner and
> numerically more stable.

In [ ]:
class MnistMLPFirst(nn.Module):
    def __init__(self, n_classes=10, p_drop=0.2):
        super().__init__()                       # required first line
        self.flatten = nn.Flatten()              # [B, 28, 28] -> [B, 784]
        self.fc1     = nn.Linear(28*28, 256)     # hidden layer 1
        self.fc2     = nn.Linear(256, 128)       # hidden layer 2
        self.out     = nn.Linear(128, n_classes) # 10 output neurons (logits)
        self.dropout = nn.Dropout(p_drop)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.out(x)                          # raw logits, NO softmax here
        return x

torch.manual_seed(SEED)
model_01 = MnistMLPFirst().to(device)
summarize_model(model_01, input_size=(1, 28, 28))

#### ⚖️ Loss, optimizer & training

The **optimizer** (`Adam`) and the **training loop** are the same as always. Only the **loss**
is task-specific: `CrossEntropyLoss` for classification.

In [ ]:
criterion_c = nn.CrossEntropyLoss()
optimizer_c = torch.optim.Adam(model_01.parameters(), lr=1e-3)
torch.manual_seed(SEED)
train_loader_c.generator.manual_seed(SEED)
history_01 = train_model(model_01, train_loader_c, val_loader_c, criterion_c, optimizer_c, n_epochs=15)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(history_01["train_loss"]) + 1), history_01["train_loss"], label="train")
plt.plot(range(1, len(history_01["val_loss"]) + 1), history_01["val_loss"],   label="validation")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.title("MNIST — training curve")
plt.legend()
plt.show()

#### 📊 Validation evaluation

Softmax converts logits into probabilities. The predicted class is their `argmax`.
Taking `argmax` directly on logits gives the same class; softmax is useful when we want probabilities.
All comparisons here use **validation** data, including the confusion matrix.

In [ ]:
logits = predict_batches(model_01, X_val_c)
probs = torch.softmax(logits, dim=1)
preds = probs.argmax(dim=1)

In [ ]:
print(probs[0])

In [ ]:
print(preds[0])

In [ ]:
nn_acc = accuracy_score(y_val_c.numpy(), preds.numpy())
print(f"ReLU network validation accuracy: {nn_acc:.3f}")
print(f"Decision Tree validation accuracy: {tree_acc:.3f}   <- baseline")

In [ ]:
cm = confusion_matrix(y_val_c.numpy(), preds.numpy(), labels=np.arange(10))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Validation confusion matrix")
plt.show()

📌 **Overfitting & the role of Dropout**

Overfitting means improving the training fit without improving generalization. Falling training
loss together with a sustained rise in validation loss is a warning sign, not proof from one epoch.
Dropout can help regularization but is not guaranteed to help every model.

Both plotted curves are measured with dropout disabled at the same checkpoint. Select the
checkpoint by validation loss, not by making the gap between the curves as small as possible.

### 🧠 Build the second architecture

This model changes **width, depth, and activation** together. Compare the two complete
architectures, but do not attribute a performance difference to Mish alone.
To isolate an activation's effect, keep widths, depth, initialization, data order, and budget fixed.

In [ ]:
class MnistMLPMish(nn.Module):
    def __init__(self, n_classes=10, p_drop=0.2):
        super().__init__()                       # required first line
        self.flatten = nn.Flatten()              # [B, 28, 28] -> [B, 784]
        self.fc1     = nn.Linear(28*28, 128)     # hidden layer 1
        self.fc2     = nn.Linear(128, 64)        # hidden layer 2
        self.fc3     = nn.Linear(64, 32)         # hidden layer 3
        self.out     = nn.Linear(32, n_classes) # 10 output neurons (logits)
        self.dropout = nn.Dropout(p_drop)

    def forward(self, x):
        x = self.flatten(x)
        x = F.mish(self.fc1(x))
        x = self.dropout(x)
        x = F.mish(self.fc2(x))
        x = self.dropout(x)
        x = F.mish(self.fc3(x))
        x = self.dropout(x)
        x = self.out(x)                          # raw logits, NO softmax here
        return x

torch.manual_seed(SEED)
model_02 = MnistMLPMish().to(device)
summarize_model(model_02, input_size=(1, 28, 28))

In [ ]:
criterion_c = nn.CrossEntropyLoss()
optimizer_c = torch.optim.Adam(model_02.parameters(), lr=1e-3)
torch.manual_seed(SEED)
train_loader_c.generator.manual_seed(SEED)
history_02 = train_model(model_02, train_loader_c, val_loader_c, criterion_c, optimizer_c, n_epochs=15)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(history_02["train_loss"]) + 1), history_02["train_loss"], label="train")
plt.plot(range(1, len(history_02["val_loss"]) + 1), history_02["val_loss"],   label="validation")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.title("MNIST Mish model — training curve")
plt.legend()
plt.show()

In [ ]:
logits_02 = predict_batches(model_02, X_val_c)
probs_02 = torch.softmax(logits_02, dim=1)
preds_02 = probs_02.argmax(dim=1)

In [ ]:
nn_acc_02 = accuracy_score(y_val_c.numpy(), preds_02.numpy())
print(f"Mish network validation accuracy: {nn_acc_02:.3f}")
# Keep candidates and their restored checkpoints for final validation-based selection.
classification_candidates = {
    "ReLU MLP": (model_01, history_01),
    "Mish MLP": (model_02, history_02),
}

# 📈 Part 2 — Regression: Auto MPG

Now the **same machinery**, but predicting a **number** instead of a class. The dataset describes
1970s–80s cars; we predict **`mpg`** (miles per gallon) from features like horsepower, weight,
and number of cylinders.

Watch how little changes: the training loop and `nn.Module` skeleton are the same — the
**input preparation**, **output layer**, **target shape/dtype**, **loss**, and **metrics** differ (remember the table 🎯).

In [ ]:
url = "https://raw.githubusercontent.com/lowoncuties/VSB-FEI-Machine-Learning-Exercises/main/datasets/auto-mpg.csv"
mpg_path = DATA_DIR / "auto-mpg.csv"
if not mpg_path.exists():
    urlretrieve(url, mpg_path)
df = pd.read_csv(mpg_path, na_values="?", sep=";")
df.head()

In [ ]:
# A few rows have a missing horsepower -> drop them (only a handful)
print("missing values before:\n", df.isna().sum()[df.isna().sum() > 0])
df = df.loc[~df.horsepower.isna(), :].copy()

### 🏷️ Handling categorical features

`origin` is a category (1=USA, 2=Europe, 3=Japan). Use this **predefined vocabulary** to
one-hot encode it; we do not infer categories from validation or test data.
We omit `car_name` to keep the exercise focused on numeric features and origin.
Its high cardinality does not mean it can never be useful.

In [ ]:
df = df.drop("car_name", axis=1)
origin_names = df["origin"].map({1: "USA", 2: "EUR", 3: "JAP"})
if origin_names.isna().any():
    raise ValueError("Unexpected or missing origin code.")
df["origin"] = pd.Categorical(origin_names, categories=["USA", "EUR", "JAP"])
df = pd.get_dummies(df, columns=["origin"], prefix="origin", dtype=float)
df.head()

In [ ]:
X = df.drop("mpg", axis=1).to_numpy(dtype=np.float32)
y = df["mpg"].to_numpy(dtype=np.float32)
if not np.isfinite(X).all() or not np.isfinite(y).all():
    raise ValueError("Features and targets must be finite after cleaning.")

# Approximately 60 / 20 / 20; integer rounding gives 234 / 79 / 79 for 392 rows.
# Retain raw features so rerunning the scaler cell cannot standardize them twice.
idx_tmp_r, idx_test_r = train_test_split(np.arange(len(X)), test_size=0.2, random_state=13)
idx_train_r, idx_val_r = train_test_split(idx_tmp_r, test_size=0.25, random_state=13)
X_train_raw, X_val_raw, X_test_raw = X[idx_train_r], X[idx_val_r], X[idx_test_r]
y_train, y_val, y_test = y[idx_train_r], y[idx_val_r], y[idx_test_r]
print("train:", X_train_raw.shape, "| val:", X_val_raw.shape, "| test:", X_test_raw.shape)

### 🔄 Scaling the features

`weight` is in the thousands, `model_year` is around 75, and cylinders take small counts.
Different scales can make gradient-based optimization harder. `StandardScaler` centers and
rescales each feature using **training statistics only**, reused for validation and test.
It does not make distributions Gaussian or guarantee convergence. Scaling the one-hot
columns is valid for this MLP, though leaving them as 0/1 is another possible design.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_loader_r = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True,
                            generator=torch.Generator().manual_seed(SEED))
val_loader_r   = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32)

### 📏 Baseline: Linear Regression

A linear model is a useful reference. In several dimensions it fits a hyperplane, not just
a single line. Compare validation errors with the neural network before final testing.

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
lin_pred = lin.predict(X_val)
print(f"Linear Regression validation -> MAE: {mean_absolute_error(y_val, lin_pred):.2f} mpg | "
      f"RMSE: {mean_squared_error(y_val, lin_pred) ** 0.5:.2f} mpg | R2: {r2_score(y_val, lin_pred):.3f}")

### 🧠 Build the regressor — spot the difference!

The hidden layers again use Linear and ReLU, with dropout after the first hidden activation.
The input width now matches the tabular features, and the output has one neuron.
**Both classifiers and this regressor have no output activation in their model definitions.**
Classification interprets its logits through cross-entropy; regression uses the raw scalar as
an MPG prediction and optimizes MSE with float targets of shape `(batch_size, 1)`.
An unrestricted output can produce negative predictions, so inspect their plausibility.

In [ ]:
class MpgMLP(nn.Module):
    def __init__(self, n_features, p_drop=0.1):
        super().__init__()
        self.fc1     = nn.Linear(n_features, 64)
        self.fc2     = nn.Linear(64, 32)
        self.out     = nn.Linear(32, 1)      # 1 output neuron
        self.dropout = nn.Dropout(p_drop)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.out(x)                      # NO activation -> real-valued output
        return x

torch.manual_seed(SEED)
model_reg = MpgMLP(n_features=X_train_tensor.shape[1]).to(device)
summarize_model(model_reg, input_size=(1, X_train_tensor.shape[1]))

In [ ]:
criterion_r = nn.MSELoss()
optimizer_r = torch.optim.Adam(model_reg.parameters(), lr=1e-2)
torch.manual_seed(SEED)
train_loader_r.generator.manual_seed(SEED)
history_reg = train_model(model_reg, train_loader_r, val_loader_r, criterion_r, optimizer_r, n_epochs=100)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(history_reg["train_loss"]) + 1), history_reg["train_loss"], label="train")
plt.plot(range(1, len(history_reg["val_loss"]) + 1), history_reg["val_loss"],   label="validation")
plt.xlabel("epoch"); plt.ylabel("MSE loss"); plt.title("Auto MPG — training curve")
plt.legend(); plt.show()

### 📊 Regression metrics

Evaluate numerical errors rather than classification accuracy:

- **MAE**: mean absolute error, in mpg.
- **RMSE**: square root of MSE, in mpg; large errors contribute more strongly through squaring.
- **MSE**: the training loss, in squared mpg.
- **$R^2 = 1 - \frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}$**:
  1 is perfect; 0 matches predicting the **evaluation set's mean**; negative values are possible.
  The usual interpretation assumes nonconstant targets. This mean is a reference in the metric,
  not a deployable baseline fitted on held-out labels.

The following results use validation data. Final test reporting comes after the activities.

In [ ]:
y_pred = predict_batches(model_reg, X_val_tensor).numpy().ravel()
mae = mean_absolute_error(y_val, y_pred)
rmse = mean_squared_error(y_val, y_pred) ** 0.5
r2 = r2_score(y_val, y_pred)
print(f"Neural network validation    -> MAE: {mae:.2f} mpg | RMSE: {rmse:.2f} mpg | R2: {r2:.3f}")
print(f"Linear Regression validation -> MAE: {mean_absolute_error(y_val, lin_pred):.2f} mpg | "
      f"RMSE: {mean_squared_error(y_val, lin_pred) ** 0.5:.2f} mpg | R2: {r2_score(y_val, lin_pred):.3f}")

In [ ]:
# Validation predictions vs reality: perfect predictions would lie on the diagonal
plt.figure(figsize=(5, 5))
sns.scatterplot(x=y_val, y=y_pred)
lims = [min(y_val.min(), y_pred.min()), max(y_val.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label="perfect")
plt.xlabel("true mpg"); plt.ylabel("predicted mpg"); plt.title("Validation predictions vs reality")
plt.legend(); plt.show()

# Activities ✏️

The code cells are intentionally empty for your solutions. Aim to complete at least three.
Use **fresh models and optimizers** for each run; reset the seed and the corresponding training
loader's generator for controlled comparisons. MNIST loaders/data have the `_c` suffix;
Auto MPG uses `train_loader_r`, `val_loader_r`, and the regression tensors.

Compare on **validation data**, keeping the test sets untouched. Record the configuration,
parameter count, validation metric, and selected epoch. One split and seed do not establish
a statistically reliable ranking. Freeze your choices before the final evaluation section.

## Activity 1 — Change the architecture 🏗️

Take **one** of the two networks — the **regressor**  *or* the **classifier** — and experiment with its architecture. Try changes below and compare the validation metric against the original.

1. **Hidden layer size** — make a layer wider or narrower.
2. **Number of layers** — add more hidden layers.
3. **Activation function** — replace `torch.relu` with another activation function.

For each variant, also look at the **model size** — the number of trainable parameters. Compare **size vs. metric**: does a bigger model actually predict better here, or just cost more parameters? 📊

Change one factor at a time if you want to isolate its effect. The two supplied classifiers
change several factors together, so their difference is not a clean ReLU-versus-Mish experiment.

## Activity 2 — Tune the Dropout rate 🎛️

Take **one** of the two networks — the **classifier** or the **regressor**  — and try `p_drop` values of `0.0`, `0.5`, `0.8`. Rebuild and retrain the model for each value, then plot the **train vs. validation** loss curves.

Which value gives the **lowest validation loss**? Inspect the curve gap as a diagnostic,
not the selection criterion: two equally poor curves can be very close. Does high dropout
cause underfitting in your run? Keep the architecture and update budget fixed.

## Activity 3 — The output-activation trap ☣️

1. Keep the target in **original MPG units** and add a sigmoid to the regressor output.
   Before training, explain why predictions restricted to `(0, 1)` cannot match these targets.
2. Train a fresh model with a linear output using the same raw targets and compare validation
   MAE in MPG units. The linear output removes that range restriction; it does not guarantee a good fit.
3. Optional: fit `MinMaxScaler` on **training targets only**, transform the targets, and try a
   sigmoid output again. Inverse-transform predictions before calculating MPG metrics.
   This is a **valid bounded-output setup**, not guaranteed to fail. Sigmoid saturation can
   slow optimization, and it cannot represent predictions outside the training target range;
   validation/test targets can transform outside `[0, 1]`. Compare with a linear output.

Use fresh models and optimizers for each variant. Keep target transformations separate from
feature scaling and do not overwrite the raw targets or original baseline model.

## Activity 4 — Transform the target with PowerTransformer 🎯

Inspect the **training** MPG histogram, then test a Yeo–Johnson `PowerTransformer` fitted
on `y_train.reshape(-1, 1)` only. Apply that fitted transform to validation targets.

- Explain why a skewed target might motivate this experiment; normality is not required by MSE.
- Retrain a fresh `MpgMLP` with a **linear output** on transformed targets.
- Inverse-transform its predictions before reporting validation MAE, RMSE, and $R^2$ in MPG units.
- Check that inverse-transformed predictions are finite; a transform can have a restricted inverse domain.
- Compare original-unit validation metrics, not MSE values on differently transformed target scales.
  Transforming the target changes the training objective and is not guaranteed to help.

If this variant is selected for final evaluation, retain its fitted target transformer and
inverse-transform the test predictions as well. Leave the original baseline model intact.

## Final held-out evaluation — run after all model choices are frozen

The defaults below choose between the two supplied classifiers using their best validation
cross-entropy and use the supplied regressor's best validation checkpoint. The tree and linear
regression are fixed reference baselines. If your activities produced a better validated candidate,
set the selection variables below to that trained model **before** running the test cell.
For a transformed-target regressor, also supply its fitted target transformer and matching
training-fitted feature preprocessing. Do not change settings after inspecting test scores.

In [ ]:
selected_classifier_name = min(
    classification_candidates,
    key=lambda name: classification_candidates[name][1]["best_val_loss"],
)
selected_classifier = classification_candidates[selected_classifier_name][0]
selected_regressor = model_reg
selected_target_transformer = None  # Supply your fitted transformer only for a transformed-target model.
selected_regression_test_features = X_test_tensor  # Must match the selected model's preprocessing.
print("Frozen classifier choice:", selected_classifier_name)

In [ ]:
# Run once after completing model selection. Do not tune against these results.
classification_test_preds = predict_batches(selected_classifier, X_test_c).argmax(dim=1).numpy()
print(f"{selected_classifier_name} test accuracy: "
      f"{accuracy_score(y_test_c.numpy(), classification_test_preds):.3f}")
tree_test_preds = tree.predict(X_test_c.reshape(-1, 28*28).numpy())
print(f"Decision Tree test accuracy: {accuracy_score(y_test_c.numpy(), tree_test_preds):.3f}")

regression_test_preds = predict_batches(selected_regressor, selected_regression_test_features).numpy()
if selected_target_transformer is not None:
    regression_test_preds = selected_target_transformer.inverse_transform(regression_test_preds)
regression_test_preds = regression_test_preds.ravel()
if not np.isfinite(regression_test_preds).all():
    raise ValueError("Non-finite predictions after inverse transformation.")
for name, prediction in [("Neural network", regression_test_preds), ("Linear Regression", lin.predict(X_test))]:
    print(f"{name} test -> MAE: {mean_absolute_error(y_test, prediction):.2f} mpg | "
          f"RMSE: {mean_squared_error(y_test, prediction) ** 0.5:.2f} mpg | "
          f"R2: {r2_score(y_test, prediction):.3f}")